# N046 · 链表反转与局部反转

**目标：** 在覆盖next前保存后继并维护已反转前缀。

**先修：** N045, N013。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** prev/curr/next；循环不变量；子区间拼接；递归对照。

## 从问题到算法

反转链表要先保存next，再修改当前指向，否则会失去未处理后缀。局部反转先固定区间前驱，将区间中后续节点逐个摘出并头插到区间开头。参数使用一基闭区间，且要求区间在链表范围内。

## 最小实现

**本章接口：** `reverse_list(head)`、`reverse_between(head, left, right)`

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def list_to_nodes(values):
    dummy = tail = ListNode()
    for value in values:
        tail.next = ListNode(value)
        tail = tail.next
    return dummy.next

def nodes_to_list(head):
    values, seen = [], set()
    while head is not None:
        if id(head) in seen:
            raise ValueError('cycle in a supposedly finite list')
        seen.add(id(head))
        values.append(head.val)
        head = head.next
    return values
def reverse_list(head):
    previous=None
    while head:
        following=head.next
        head.next=previous
        previous,head=head,following
    return previous

def reverse_between(head,left,right):
    if not 1<=left<=right: raise ValueError('requires 1 <= left <= right')
    dummy=ListNode(0,head); before=dummy
    for _ in range(left-1): before=before.next
    tail=before.next
    for _ in range(right-left):
        moved=tail.next; tail.next=moved.next
        moved.next=before.next; before.next=moved
    return dummy.next

## 正确性、前提与复杂度

previous指向原前缀的逆序链，head指向尚未处理后缀，二者节点不交且并集不变。局部头插保留区间尾tail，不改变区间外连接。

**代价：** 整链与局部反转 O(n)时间、O(1)辅助空间；不创建新数据节点，只有局部哨兵。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

h=list_to_nodes([1,2,3,4]); previous=None; rows=[]
while h:
    following=h.next; h.next=previous; previous,h=h,following
    rows.append((nodes_to_list(previous),nodes_to_list(h)))
show_table(['已反转前缀','未处理后缀'],rows)

已反转前缀,未处理后缀
[1],"[2, 3, 4]"
"[2, 1]","[3, 4]"
"[3, 2, 1]",[4]
"[4, 3, 2, 1]",[]


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert nodes_to_list(reverse_list(list_to_nodes([1,2,3])))==[3,2,1]
assert nodes_to_list(reverse_between(list_to_nodes([1,2,3,4,5]),2,4))==[1,4,3,2,5]
h=list_to_nodes([1,2,3,4]); original=[]; p=h
while p: original.append(p); p=p.next
h=reverse_list(reverse_list(h)); restored=[]; p=h
while p: restored.append(p); p=p.next
assert restored==original
for n in range(1,8):
    for l in range(1,n+1):
        for r in range(l,n+1):
            a=list(range(n)); assert nodes_to_list(reverse_between(list_to_nodes(a),l,r))==a[:l-1]+a[l-1:r][::-1]+a[r:]
print('N046: 所有本章断言通过')

N046: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 写递归反转并计入调用栈。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 局部反转后验证两端拼接。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 206. Reverse Linked List（canonical）：[官方入口](https://leetcode.com/problems/reverse-linked-list/)
- 92. Reverse Linked List II（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。